In [ ]:
from datetime import datetime
import pandas as pd

# 1. APNA FILE PATH YAHA CHECK KARLO
file_path = (
    r"C:\Users\user\OneDrive\Desktop\NbJp\Stocks-Analysis-System\results\signals_log.csv"
)


def generate_overall_dashboard(csv_path):
    try:
        # CSV file ko read karein
        df = pd.read_csv(csv_path)

        if df.empty:
            print("❌ CSV file khali hai!")
            return

        # Data Cleaning: values ko numeric mein convert karein
        df["pips_result"] = pd.to_numeric(df["pips_result"], errors="coerce")

        # --- CORE METRICS CALCULATIONS ---
        total_trades = len(df)
        tp_count = len(df[df["outcome"] == "TP"])
        sl_count = len(df[df["outcome"] == "SL"])

        # Win Rate %
        win_rate = (tp_count / total_trades) * 100 if total_trades > 0 else 0

        # Total, Average, Max Pips
        net_pips = df["pips_result"].sum()
        avg_pips_per_trade = df["pips_result"].mean()
        max_win = df["pips_result"].max()
        max_loss = df["pips_result"].min()

        # Streak Calculations (Consecutive Wins/Losses)
        current_streak = 0
        max_win_streak = 0
        max_loss_streak = 0
        last_outcome = None

        for outcome in df["outcome"]:
            if outcome == "TP":
                if last_outcome == "TP":
                    current_streak += 1
                else:
                    current_streak = 1
                max_win_streak = max(max_win_streak, current_streak)
            elif outcome == "SL":
                if last_outcome == "SL":
                    current_streak += 1
                else:
                    current_streak = 1
                max_loss_streak = max(max_loss_streak, current_streak)
            last_outcome = outcome

        # --- DASHBOARD OUTPUT PRINTING ---
        print("=" * 60)
        print("                 🏆 OVERALL TRADING DASHBOARD               ")
        print("=" * 60)

        print(f"📊 GENERAL METRICS:")
        print(f"   🔹 Total Trades Executed : {total_trades}")
        print(f"   🟩 Total Profit (TP) Hits: {tp_count}")
        print(f"   🟥 Total Loss (SL) Hits  : {sl_count}")
        print(f"   🎯 Lifetime Win Rate     : {win_rate:.2f}%")

        print(f"\n🪙 PIPS PERFORMANCE:")
        print(f"   🟢 Net Pips Gained       : {net_pips:+.1f} pips")
        print(f"   🔵 Avg Pips Per Trade    : {avg_pips_per_trade:+.1f} pips")
        print(f"   ⭐ Biggest Win (Max TP)  : {max_win:+.1f} pips")
        print(f"   ⚠️ Biggest Loss (Max SL) : {max_loss:.1f} pips")

        print(f"\n🔥 STREAK STATISTICS:")
        print(f"   🚀 Maximum Win Streak    : {max_win_streak} consecutive TPs")
        print(f"   📉 Maximum Loss Streak   : {max_loss_streak} consecutive SLs")

        print("=" * 60)

    except FileNotFoundError:
        print(f"❌ Error: File is path par nahi mili: {csv_path}")
    except Exception as e:
        print(f"❌ Kuch error aaya: {str(e)}")


if __name__ == "__main__":
    generate_overall_dashboard(file_path)

In [3]:
from datetime import datetime
import csv
import os
import re

# 1. FILE PATH
file_path = (
    r"C:\Users\user\OneDrive\Desktop\NbJp\Stocks-Analysis-System\results\signals_log.csv"
)

# 2. ISME AAP EK SIGNAL YA SAARE SIGNALS EK SAATH PASTE KAR SAKTE HAIN
raw_signals_batch = """
📅 Tuesday, 02 Jun 2026 | 🕐 07:32:09 UTC
[   4] ⭐ BUY XAUUSD
Entry: 4534.860
━━━━━━━━━━━━━━━━
SL1: 4529.860 (-50 pips)
SL2: 4528.706 (-62 pips)
━━━━━━━━━━━━━━━━
TP1: 4544.860 (+100 pips)
TP2: 4547.167 (+123 pips)
━━━━━━━━━━━━━━━━
Conf: 73% | TP: 70% | ADX: 53.2
RSI: 62.5 | TFs: 3/5
Pred: +0.0002
━━━━━━━━━━━━━━━━
📰 News: Clear
😐 Neutral (50)
━━━━━━━━━━━━━━━━
🔬 Beta Testing — Ahmed R. Hussain sl2
"""


def process_and_append_batch(text_batch):
    # Signals ko split karne ke liye calendar emoji use kiya hai
    blocks = text_batch.strip().split("📅")
    rows_to_add = []

    for block in blocks:
        if not block.strip() or "Entry:" not in block:
            continue

        # 1. Timestamp Parsing
        date_match = re.search(
            r"([^|]+)\|\s*🕐\s*([\d:]+)", block, re.IGNORECASE
        )
        if date_match:
            date_clean = date_match.group(1).split(",")[-1].strip()
            time_str = date_match.group(2).strip()
            dt_obj = datetime.strptime(
                f"{date_clean} {time_str}", "%d %b %Y %H:%M:%S"
            )
            timestamp = f"{dt_obj.month}/{dt_obj.day}/{dt_obj.year} {dt_obj.hour}:{dt_obj.minute:02d}"
        else:
            timestamp = datetime.now().strftime("%m/%d/%Y %H:%M")

        # 2. Signal details extraction
        asset_match = re.search(r"⭐\s*(BUY|SELL)\s*(\w+)", block)
        signal = asset_match.group(1) if asset_match else "SELL"
        asset = asset_match.group(2) if asset_match else "XAUUSD"

        entry = float(re.search(r"Entry:\s*([\d.]+)", block).group(1))
        sl1 = float(re.search(r"SL1:\s*([\d.]+)", block).group(1))
        sl2 = float(re.search(r"SL2:\s*([\d.]+)", block).group(1))
        tp1 = float(re.search(r"TP1:\s*([\d.]+)", block).group(1))
        tp2 = float(re.search(r"TP2:\s*([\d.]+)", block).group(1))

        pips_sl1 = abs(
            int(re.search(r"SL1:.*?\((-?\d+)\s*pips\)", block).group(1))
        )
        pips_sl2 = abs(
            int(re.search(r"SL2:.*?\((-?\d+)\s*pips\)", block).group(1))
        )
        pips_tp1 = abs(
            int(re.search(r"TP1:.*?\(\+?(-?\d+)\s*pips\)", block).group(1))
        )
        pips_tp2 = abs(
            int(re.search(r"TP2:.*?\(\+?(-?\d+)\s*pips\)", block).group(1))
        )

        conf = int(re.search(r"Conf:\s*(\d+)%", block).group(1))
        tp_prob = int(re.search(r"TP:\s*(\d+)%", block).group(1))
        adx = float(re.search(r"ADX:\s*([\d.]+)", block).group(1))

        pred_match = re.search(r"Pred:\s*(-?[\d.]+)", block)
        pred = float(pred_match.group(1)) if pred_match else 0.0

        # 3. AUTOMATIC OUTCOME DETECTION FROM TEXT ENDING
        last_lines = block.lower().strip().split("\n")[-2:]
        outcome_text = " ".join(last_lines)

        if "sl" in outcome_text:
            outcome = "SL"
            # Agar log me clear mention hai ya text [ 31] hai toh SL2 setup karega
            if "sl2" in outcome_text or "[  31]" in block:
                exit_price = sl2
                pips_result = -pips_sl2
                notes = f"SL2 hit -{pips_sl2}"
            else:
                exit_price = sl1
                pips_result = -pips_sl1
                notes = f"SL1 hit -{pips_sl1}"
        elif "tp 2" in outcome_text or "tp2" in outcome_text:
            outcome = "TP"
            exit_price = tp2
            pips_result = float(pips_tp2)  # <-- Dono ko plus nahi karna, sirf TP2 lena hai
            notes = f"TP2 hit +{pips_tp2}"
        elif "tp 1" in outcome_text or "tp1" in outcome_text:
            outcome = "TP"
            exit_price = tp1
            pips_result = float(pips_tp1)
            notes = f"TP1 hit +{pips_tp1}"
        else:
            # Agar sirf generic "Tp" likha ho (Jaise Log 1 mein hai)
            outcome = "TP"
            exit_price = tp1
            pips_result = float(pips_tp1)
            notes = f"TP1 hit +{pips_tp1}"

        row_data = {
            "timestamp": timestamp,
            "asset": asset,
            "timeframe": "15min",
            "signal": signal,
            "entry": entry,
            "sl1": sl1,
            "sl2": sl2,
            "tp1": tp1,
            "tp2": tp2,
            "pips_sl1": pips_sl1,
            "pips_sl2": pips_sl2,
            "pips_tp1": pips_tp1,
            "pips_tp2": pips_tp2,
            "rr_ratio": 2,
            "pred": pred,
            "adx": adx,
            "confidence": conf,
            "tp_prob": tp_prob,
            "outcome": outcome,
            "exit_price": exit_price,
            "pips_result": pips_result,
            "notes": notes,
        }
        rows_to_add.append(row_data)

    # 4. CHRONOLOGICAL SORTING (Purane trade pehle aur naye baad me save honge)
    rows_to_add.sort(
        key=lambda x: datetime.strptime(x["timestamp"], "%m/%d/%Y %H:%M")
    )

    # 5. CSV APPEND LOGIC
    if not rows_to_add:
        print("❌ Koi valid signal nahi mila parse karne ke liye.")
        return

    fieldnames = list(rows_to_add[0].keys())
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    file_exists = os.path.exists(file_path)

    with open(file_path, mode="a", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows_to_add)

    print(
        f"✔️ Success! {len(rows_to_add)} rows automatically process karke file me append kar di gayi hain."
    )


# Execution
if __name__ == "__main__":
    process_and_append_batch(raw_signals_batch)

✔️ Success! 1 rows automatically process karke file me append kar di gayi hain.


In [4]:
from datetime import datetime
import pandas as pd

# File Path
file_path = (
    r"C:\Users\user\OneDrive\Desktop\NbJp\Stocks-Analysis-System\results\signals_log.csv"
)

# Aaj ki date setup (2nd June 2026)
todays_date = "6/2/2026"


def generate_daily_summary(csv_path, target_date):
    try:
        # Read the CSV file
        df = pd.read_csv(csv_path)

        # Convert timestamp column to string and filter by today's date
        df["timestamp"] = df["timestamp"].astype(str)
        df_today = df[df["timestamp"].str.startswith(target_date)].copy()

        if df_today.empty:
            print(f"❌ {target_date} ke liye koi data nahi mila file mein.")
            return

        # Core Metrics Calculations
        total_trades = len(df_today)
        tp_trades = len(df_today[df_today["outcome"] == "TP"])
        sl_trades = len(df_today[df_today["outcome"] == "SL"])
        win_rate = (tp_trades / total_trades) * 100 if total_trades > 0 else 0
        total_pips = df_today["pips_result"].sum()

        # Display Summary
        print("=" * 50)
        print(f"📊 TRADING SUMMARY FOR: {target_date}")
        print("=" * 50)
        print(f"📈 Total Trades Executed : {total_trades}")
        print(f"✅ Take Profit (TP) Hits : {tp_trades}")
        print(f"❌ Stop Loss (SL) Hits   : {sl_trades}")
        print(f"🎯 Win Rate              : {win_rate:.2f}%")
        print(f"🟢 Total Pips Gained     : {total_pips:+.1f} pips")
        print("=" * 50)
        print("\n📝 INDIVIDUAL TRADE BREAKDOWN:")
        print("-" * 50)

        # Print row-by-row matrix
        for idx, row in df_today.iterrows():
            time_part = row["timestamp"].split(" ")[-1]
            print(
                f"⏰ Time: {time_part} | {row['signal']} {row['asset']} | Outcome: {row['outcome']} | Pips: {row['pips_result']:+g} | ({row['notes']})"
            )
        print("=" * 50)

    except FileNotFoundError:
        print(f"❌ Error: File nahi mili is path par: {csv_path}")
    except Exception as e:
        print(f"❌ Kuch error aaya: {str(e)}")


if __name__ == "__main__":
    generate_daily_summary(file_path, todays_date)

📊 TRADING SUMMARY FOR: 6/2/2026
📈 Total Trades Executed : 5
✅ Take Profit (TP) Hits : 4
❌ Stop Loss (SL) Hits   : 1
🎯 Win Rate              : 80.00%
🟢 Total Pips Gained     : +367.0 pips

📝 INDIVIDUAL TRADE BREAKDOWN:
--------------------------------------------------
⏰ Time: 5:26 | BUY XAUUSD | Outcome: TP | Pips: +107 | (TP2 hit +107)
⏰ Time: 5:44 | BUY XAUUSD | Outcome: TP | Pips: +100 | (TP1 hit +100)
⏰ Time: 6:16 | BUY XAUUSD | Outcome: TP | Pips: +122 | (TP2 hit +122)
⏰ Time: 6:42 | BUY XAUUSD | Outcome: TP | Pips: +100 | (TP1 hit +100)
⏰ Time: 7:32 | BUY XAUUSD | Outcome: SL | Pips: -62 | (SL2 hit -62)
